In [1]:
def read_dataframe(filename):
    columns = [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance"
    ]

    df = pd.read_parquet(filename, columns=columns)
    df=df.head(1000)  # For testing purposes, limit to first 1000 rows
    df["duration"] = (
        df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    ).dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ["PULocationID", "DOLocationID"]
    numerical = ["trip_distance"]

    df[categorical] = df[categorical].astype(str)

    return df

In [5]:
import numpy as np
import pickle

In [4]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("new")

2026/09/13 14:11:16 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/13 14:11:16 INFO mlflow.store.db.utils: Updating database tables
2026/09/13 14:11:18 INFO mlflow.tracking.fluent: Experiment with name 'new' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/MLOPs/03-pipeline/mlruns/1', creation_time=1789308678096, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789308678096, lifecycle_stage='active', name='new', tags={}, trace_location=None, workspace='default'>

In [6]:
df_train=read_dataframe("../01-intro/yellow_tripdata_2026-01.parquet")

In [7]:
categorical=["PULocationID","DOLocationID"]
numerical=["trip_distance"]
dv=DictVectorizer()
train_dict=df_train[categorical+numerical].to_dict(orient="records")
X_train=dv.fit_transform(train_dict)


In [8]:
y_train=df_train["duration"]

In [9]:
X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

In [8]:
with mlflow.start_run():
    mlflow.set_tag("developper","hakim")
    alpha=.002
    mlflow.log_param("alpha",alpha)
    lr=Lasso(alpha=alpha)
    lr.fit(X_train,y_train)
    y_pred=lr.predict(X_train)
    mae=mean_absolute_error(y_train,y_pred)
    mlflow.log_metric("mae",mae)
    mlflow.log_artifact(local_path="../models/lin_reg.bin",artifact_path="models_pickle")

In [9]:
with open("../models/lin_reg.bin","wb") as f_out:
    pickle.dump((dv,lr),f_out)

In [12]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model","xgboost")
        mlflow.log_params(params)

        
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=xgb.DMatrix(X_train, label=y_train),
            num_boost_round=100,
            evals=[(xgb.DMatrix(X_train, label=y_train), "train")],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(xgb.DMatrix(X_train))
        mae = mean_absolute_error(y_train, y_pred)


        mlflow.log_metric("mae", mae)
        mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

    return {"loss": mae, "status": STATUS_OK}

In [13]:
search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 4, 100, 1)),
    "learning_rate": hp.loguniform("learning_rate", -3, 0),
    "reg_alpha": hp.loguniform("reg_alpha", -5, -1),
    "reg_lambda": hp.loguniform("reg_lambda", -6, -1),
    "min_child_weight": hp.loguniform("min_child_weight", -1, 3),
    "objective": "reg:squarederror",
}
best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials(),
)

[0]	train-rmse:5.73365                                
[1]	train-rmse:4.36143                                
[2]	train-rmse:3.86888                                
[3]	train-rmse:3.56304                                
[4]	train-rmse:3.39230                                
[5]	train-rmse:3.26129                                
[6]	train-rmse:3.10216                                
[7]	train-rmse:2.97632                                
[8]	train-rmse:2.86030                                
[9]	train-rmse:2.73578                                
[10]	train-rmse:2.64182                               
[11]	train-rmse:2.52588                               
[12]	train-rmse:2.43987                               
[13]	train-rmse:2.37302                               
[14]	train-rmse:2.30826                               
[15]	train-rmse:2.24555                               
[16]	train-rmse:2.18756                               
[17]	train-rmse:2.13433                               
[18]	train

2026/09/13 14:17:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



[0]	train-rmse:10.18417                                                          
[1]	train-rmse:9.81782                                                           
[2]	train-rmse:9.47763                                                           
[3]	train-rmse:9.16291                                                           
[4]	train-rmse:8.87075                                                           
[5]	train-rmse:8.60191                                                           
[6]	train-rmse:8.35457                                                           
[7]	train-rmse:8.12693                                                           
[8]	train-rmse:7.91356                                                           
[9]	train-rmse:7.71974                                                           
[10]	train-rmse:7.53897                                                          
[11]	train-rmse:7.37590                                                          
[12]	train-rmse:

2026/09/13 14:17:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



[0]	train-rmse:9.28796                                                           
[1]	train-rmse:8.19811                                                           
[2]	train-rmse:7.28427                                                           
[3]	train-rmse:6.47474                                                           
[4]	train-rmse:5.80201                                                           
[5]	train-rmse:5.21270                                                           
[6]	train-rmse:4.72647                                                           
[7]	train-rmse:4.30380                                                           
[8]	train-rmse:3.94632                                                           
[9]	train-rmse:3.64456                                                           
[10]	train-rmse:3.37843                                                          
[11]	train-rmse:3.13205                                                          
[12]	train-rmse:

...

2026/09/13 14:18:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



[0]	train-rmse:8.26694                                                            
[1]	train-rmse:6.58039                                                            
[2]	train-rmse:5.32732                                                            
[3]	train-rmse:4.42250                                                            
[4]	train-rmse:3.75691                                                            
[5]	train-rmse:3.24297                                                            
[6]	train-rmse:2.88448                                                            
[7]	train-rmse:2.59070                                                            
[8]	train-rmse:2.35151                                                            
[9]	train-rmse:2.13590                                                            
[10]	train-rmse:1.97028                                                           
[11]	train-rmse:1.81445                                                           
[12]

2026/09/13 14:18:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



[0]	train-rmse:9.74504                                                            
[1]	train-rmse:9.00601                                                            
[2]	train-rmse:8.35175                                                            
[3]	train-rmse:7.76933                                                            
[4]	train-rmse:7.24483                                                            
[5]	train-rmse:6.77203                                                            
[6]	train-rmse:6.36743                                                            
[7]	train-rmse:5.97996                                                            
[8]	train-rmse:5.65304                                                            
[9]	train-rmse:5.35975                                                            
[10]	train-rmse:5.08311                                                           
[11]	train-rmse:4.85801                                                           
[12]

KeyboardInterrupt: 

In [17]:
with mlflow.start_run():
        mlflow.set_tag("model","xgboost")

        best_params = {
            "learning_rate": 0.9919286966376871,
            "max_depth": 67,
            "min_child_weight": 0.8916854838870615,
            "objective": "reg:squarederror",
            "reg_alpha": 0.008594124238282383,
            "reg_lambda": 0.042113157151565286
        }
        mlflow.log_params(best_params)
        booster = xgb.train(
            params=best_params,
            dtrain=xgb.DMatrix(X_train, label=y_train),
            num_boost_round=100,
            evals=[(xgb.DMatrix(X_train, label=y_train), "train")],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(xgb.DMatrix(X_train))
        mae = mean_absolute_error(y_train, y_pred)

        with open("../models/xgb_model.bin", "wb") as f_out:
            pickle.dump(dv, f_out)
        mlflow.log_metric("mae", mae)
        mlflow.log_artifact(local_path="../models/xgb_model.bin", artifact_path="models_pickle")
        mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")    

[0]	train-rmse:0.82795
[1]	train-rmse:0.17074
[2]	train-rmse:0.15878
[3]	train-rmse:0.15852
[4]	train-rmse:0.15847
[5]	train-rmse:0.15846
[6]	train-rmse:0.15845
[7]	train-rmse:0.15844
[8]	train-rmse:0.15844
[9]	train-rmse:0.15844
[10]	train-rmse:0.15844
[11]	train-rmse:0.15844
[12]	train-rmse:0.15844
[13]	train-rmse:0.15844
[14]	train-rmse:0.15844
[15]	train-rmse:0.15844
[16]	train-rmse:0.15844
[17]	train-rmse:0.15844
[18]	train-rmse:0.15844
[19]	train-rmse:0.15844
[20]	train-rmse:0.15844
[21]	train-rmse:0.15844
[22]	train-rmse:0.15844
[23]	train-rmse:0.15844
[24]	train-rmse:0.15844
[25]	train-rmse:0.15844
[26]	train-rmse:0.15844
[27]	train-rmse:0.15844
[28]	train-rmse:0.15844
[29]	train-rmse:0.15844
[30]	train-rmse:0.15844
[31]	train-rmse:0.15844
[32]	train-rmse:0.15844
[33]	train-rmse:0.15844
[34]	train-rmse:0.15844
[35]	train-rmse:0.15844
[36]	train-rmse:0.15844
[37]	train-rmse:0.15844
[38]	train-rmse:0.15844
[39]	train-rmse:0.15844
[40]	train-rmse:0.15844
[41]	train-rmse:0.15844
[4

2026/09/12 10:12:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [16]:
mlflow.autolog(disable=True)

In [18]:
logged_model="runs:/ae2c0d339ba94611843849ed4e45a1f6/models_mlflow"
loaded_model = mlflow.pyfunc.load_model(logged_model)

In [19]:
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: /workspaces/MLOPs/01-intro/mlruns/1/models/m-fd380c13205d4a6db6e17d956bde7363/artifacts
  flavor: mlflow.xgboost
  run_id: ae2c0d339ba94611843849ed4e45a1f6

In [20]:
xgboost_model=mlflow.xgboost.load_model(logged_model)


In [21]:
xgboost_model

In [22]:
xgboost_model.predict(xgb.DMatrix(X_train))

array([ 5.5549235,  5.7171426,  8.881725 , 42.799553 , 13.500369 ,
       13.599484 , 10.630138 , 24.616564 , 37.7329   ,  9.582433 ,
       36.199486 , 12.483995 , 27.682947 ,  1.9506497,  4.617155 ,
       21.933325 , 26.26826  , 41.953976 ,  9.966627 ,  6.5646887,
        4.6831927,  4.2160606, 22.551907 ,  8.04829  ,  7.916733 ,
       38.182915 ,  5.182784 ,  6.7282143, 15.598589 , 15.18268  ,
       18.73763  ,  9.382558 , 12.065534 , 17.833652 ,  7.2997913,
       25.583742 , 19.546734 , 11.549755 , 46.95081  , 30.433306 ,
        7.90009  , 10.883348 , 50.729885 ,  4.9009395, 23.032583 ,
       17.580948 ,  6.187775 ,  5.2827206,  6.8288093, 16.54735  ,
       43.048958 , 53.81189  , 25.282436 , 21.084345 , 16.283693 ,
       22.366589 , 10.902911 , 21.333271 , 38.115543 , 14.030471 ,
        8.600625 , 11.717032 ,  4.1377935, 25.848907 , 26.151518 ,
       23.749956 , 30.449903 , 24.349392 , 30.398878 , 24.016499 ,
       39.949894 , 27.500584 ,  8.298139 ,  2.5165095,  2.5165